In [1]:
# cell 1
!pip -q install -U transformers accelerate peft sentencepiece scikit-learn pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 72.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 90.1 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.


In [2]:
# Cell 16 — Drive setup (no CONFIG here)
from google.colab import drive
import os, shutil, glob

drive.mount("/content/drive")

DRIVE_ART_DIR = "/content/drive/MyDrive/liar_runs/artifacts_liar_v1"
os.makedirs(DRIVE_ART_DIR, exist_ok=True)

# 可选：把本地 artifacts 拷到 Drive，避免丢
LOCAL_ART_DIR = "/content/artifacts_liar_v1"
if os.path.exists(LOCAL_ART_DIR):
    for p in glob.glob(os.path.join(LOCAL_ART_DIR, "*")):
        name = os.path.basename(p)
        dst = os.path.join(DRIVE_ART_DIR, name)
        if os.path.isdir(p):
            if not os.path.exists(dst):
                shutil.copytree(p, dst)
        else:
            shutil.copy2(p, dst)

print("✅ DRIVE_ART_DIR =", DRIVE_ART_DIR)
print("Drive manifests:", len(glob.glob(os.path.join(DRIVE_ART_DIR, "manifest_*.json"))))


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ DRIVE_ART_DIR = /content/drive/MyDrive/liar_runs/artifacts_liar_v1
Drive manifests: 7


In [3]:
# cell 2
import os, time, json, random
import numpy as np
import pandas as pd
import torch

from sklearn.metrics import f1_score, accuracy_score
from transformers import (
    AutoTokenizer, AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer, Seq2SeqTrainingArguments
)

# ✅ 只在这里做最小增量：加 LoraConfig
from peft import (
    PromptTuningConfig, PromptTuningInit,
    LoraConfig,
    get_peft_model, TaskType, PeftModel
)

# ✅ 防呆：确保你在 cell2 之前已经定义了 DRIVE_ART_DIR
assert "DRIVE_ART_DIR" in globals(), (
    "❌ DRIVE_ART_DIR 还没定义！请先运行你挂载 Drive + 设置 DRIVE_ART_DIR 的 cell（例如你之前的 cell16）。"
)

CONFIG = {
    # ===== 你主要改这两行 =====
    #"model_name": "google/flan-t5-small",   # 也可以: "facebook/bart-base"
    "model_name": "facebook/bart-base",
    "run_type": "instruction",              # "instruction" | "cot" | "prompt"

    # ===== NEW: 选择微调方式（默认 full，不影响你当前 baseline）=====
    # "full"  : 全参 fine-tuning（你的 instruction baseline）
    # "prompt": prompt tuning（你 vt 那条线）
    # "lora"  : LoRA（我们要加的第三条线）
    "finetune_method": "lora",              # "full" | "prompt" | "lora"

    # ===== 训练超参 =====
    "seed": 42,
    "epochs": 3,
    "lr": 5e-4,  # 2e-4 / 5e-5 1e-3
    "train_bs": 8,
    "eval_bs": 16,
    "grad_accum": 4,

    # ===== 长度与解码（统一评测很重要） =====
    "max_input_len": 256,
    "max_target_len": 64,
    "max_new_tokens": 4,   # ✅ 你现在一直强调 cap=4，就把这里当作 cap

    # ===== prompt tuning 相关 =====
    "num_virtual_tokens": 0,

    # ===== NEW: LoRA 超参（只有 finetune_method="lora" 才会用到）=====
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    # BART 常用：q_proj, v_proj；FLAN-T5 常用：q, v（如果你后面跑 T5，我再给你一份 target_modules 列表）
    "lora_target_modules": ["q_proj", "v_proj"],

    # ===== 方便你先 smoke test（别一上来跑全量） =====
    "train_limit": 8000,    # None 表示全量
    "valid_limit": 1000,
    "test_limit": 2000,

    # ===== 输出 =====
    "artifacts_dir": DRIVE_ART_DIR,
    "fp16": True,  # 有 GPU 时会启用
}

# Cell 2（在 CONFIG 定义完后追加）
print("artifacts_dir =", CONFIG["artifacts_dir"])
assert "drive/MyDrive" in CONFIG["artifacts_dir"], (
    "❌ artifacts_dir 不是 Drive 路径！请先在挂载 Drive 的 cell 里设定。"
)

os.makedirs(CONFIG["artifacts_dir"], exist_ok=True)

def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(CONFIG["seed"])
print("CONFIG loaded:", CONFIG)


artifacts_dir = /content/drive/MyDrive/liar_runs/artifacts_liar_v1
CONFIG loaded: {'model_name': 'facebook/bart-base', 'run_type': 'instruction', 'finetune_method': 'lora', 'seed': 42, 'epochs': 3, 'lr': 0.0005, 'train_bs': 8, 'eval_bs': 16, 'grad_accum': 4, 'max_input_len': 256, 'max_target_len': 64, 'max_new_tokens': 4, 'num_virtual_tokens': 0, 'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05, 'lora_target_modules': ['q_proj', 'v_proj'], 'train_limit': 8000, 'valid_limit': 1000, 'test_limit': 2000, 'artifacts_dir': '/content/drive/MyDrive/liar_runs/artifacts_liar_v1', 'fp16': True}


In [4]:
# cell3
import pandas as pd
import os

# 下载并解压 LIAR (如果不存在的话)
if not os.path.exists("liar_dataset.zip"):
    !wget -q https://www.cs.ucsb.edu/~william/data/liar_dataset.zip -O liar_dataset.zip
    !unzip -q -o liar_dataset.zip

colnames = [
    "id", "label", "statement", "subject", "speaker", "job_title",
    "state_info", "party_affiliation",
    "barely_true_counts", "false_counts", "half_true_counts",
    "mostly_true_counts", "pants_on_fire_counts",
    "context"
]

# 读取数据
train_df = pd.read_csv("train.tsv", sep="\t", header=None, names=colnames, quoting=3)
valid_df = pd.read_csv("valid.tsv", sep="\t", header=None, names=colnames, quoting=3)
test_df  = pd.read_csv("test.tsv",  sep="\t", header=None, names=colnames, quoting=3)

false_set = {"pants-fire", "false", "barely-true"}
true_set  = {"half-true", "mostly-true", "true"}

def to_binary(df: pd.DataFrame, strict=True) -> pd.DataFrame:
    """
    labels: 1=True, 0=False
    如要改二分规则：改这里
    """
    df = df.copy()
    df["label_norm"] = df["label"].astype(str).str.strip().str.lower()
    if strict:
        allowed = false_set | true_set
        unknown = set(df["label_norm"].unique()) - allowed
        # assert len(unknown) == 0, f"Unknown labels: {unknown}" # 某些脏数据可能会报错，暂时注释掉

    df["labels"] = df["label_norm"].apply(lambda s: 1 if s in true_set else 0)
    # 保留 statement 作为输入文本
    return df[["statement", "labels"]]

train_b = to_binary(train_df)
valid_b = to_binary(valid_df)
test_b  = to_binary(test_df)

# --- 【新增功能】 数据采样工具 (用于 Low-Resource 实验) ---
def sample_dataset(df, ratio=1.0, seed=42):
    """
    根据比例采样数据集。
    ratio=0.1 表示只用 10% 的数据。
    """
    if ratio >= 1.0:
        return df
    return df.sample(frac=ratio, random_state=seed).reset_index(drop=True)
# ----------------------------------------------------

# 原有的 limit 逻辑 (保留以兼容你之前的 CONFIG 设置)
def apply_limit(df, limit):
    if limit is None:
        return df
    return df.sample(n=min(limit, len(df)), random_state=CONFIG.get("seed", 42)).reset_index(drop=True)

# 应用 CONFIG 里的限制（如果有的话）
train_b = apply_limit(train_b, CONFIG.get("train_limit", None))
valid_b = apply_limit(valid_b, CONFIG.get("valid_limit", None))
test_b  = apply_limit(test_b,  CONFIG.get("test_limit", None))

# 【关键】统一变量名，方便后续实验循环调用
df_train = train_b
df_valid = valid_b
df_test  = test_b

print("Dataset Loaded & Processed:")
print(f"Train Size: {len(df_train)}")
print(f"Valid Size: {len(df_valid)}")
print(f"Test  Size: {len(df_test)}")
print("Train Label Balance:", df_train["labels"].value_counts().to_dict())

df_train.head(3)

Dataset Loaded & Processed:
Train Size: 8000
Valid Size: 1000
Test  Size: 1283
Train Label Balance: {1: 4532, 0: 3468}


,statement,labels
0,The Capitol was built by slaves.,1
1,While fat-cat bureaucrats at the Department of...,1
2,U.S. Rep. Allen West wants to bring back earma...,0


In [5]:
# Cell 4
import re

def label_to_str(y: int) -> str:
    # 1=True, 0=False
    return "1" if int(y) == 1 else "0"

def build_prompt(statement: str, run_type: str) -> str:
    statement = str(statement)

    if run_type in ("instruction", "prompt"):
        return (
            "You are a careful fact-checking assistant.\n"
            "Task: Judge whether the following claim is true or false.\n"
            "Output format:\n"
            "- Output ONLY one token: 1 (true) or 0 (false).\n\n"
            f'Statement: "{statement}"\n'
            "Answer:"
        )

    if run_type == "cot":
        return (
            "You are a careful fact-checking assistant.\n"
            "Task: Judge whether the following claim is true or false.\n"
            "Output format (must follow exactly):\n"
            "Reasoning: ...\n"
            "Final label: 1 or 0\n\n"
            f'Statement: "{statement}"\n'
            "Answer:"
        )

    raise ValueError("run_type must be instruction|cot|prompt")

def format_target(y: int, run_type: str) -> str:
    lab = label_to_str(y)
    if run_type == "cot":
        return f"Reasoning: Based on general knowledge, the claim is assessed.\nFinal label: {lab}"
    return lab

# ===== 解析 0/1（替换你原来的 _FINAL_RE/_TF_RE + extract_label）=====
_FINAL01_RE = re.compile(r"final\s*label\s*:\s*([01])\b", re.IGNORECASE)
_DIGIT_RE = re.compile(r"\b([01])\b")

def extract_label(generated_text: str) -> int:
    """
    - 优先抓 CoT 的 Final label: 0/1
    - 否则抓第一个独立的 0 或 1
    - 找不到就默认 0
    """
    s = str(generated_text).strip()
    m = _FINAL01_RE.search(s)
    if m:
        return int(m.group(1))
    m2 = _DIGIT_RE.search(s)
    if m2:
        return int(m2.group(1))
    return 0


In [6]:
# Cell 5
from datasets import Dataset

assert CONFIG["run_type"] in ["instruction", "prompt", "cot"], f"Unexpected run_type: {CONFIG['run_type']}"

def make_hf_dataset(df: pd.DataFrame, run_type: str) -> Dataset:
    df = df.copy()
    df["statement"] = df["statement"].astype(str)
    df["labels"] = df["labels"].astype(int)

    rows = {
        "input_text": [build_prompt(s, run_type) for s in df["statement"].tolist()],
        "target_text": [format_target(y, run_type) for y in df["labels"].tolist()],
        "labels": df["labels"].tolist(),  # keep for debug; tokenized dataset will drop it later
    }
    return Dataset.from_dict(rows)

train_ds = make_hf_dataset(train_b, CONFIG["run_type"])
valid_ds = make_hf_dataset(valid_b, CONFIG["run_type"])
test_ds  = make_hf_dataset(test_b,  CONFIG["run_type"])

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"], use_fast=True)

def tokenize_batch(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=CONFIG["max_input_len"],
        truncation=True,
    )
    labels = tokenizer(
        text_target=batch["target_text"],
        max_length=CONFIG["max_target_len"],
        truncation=True,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

train_tok = train_ds.map(tokenize_batch, batched=True, remove_columns=train_ds.column_names)
valid_tok = valid_ds.map(tokenize_batch, batched=True, remove_columns=valid_ds.column_names)
test_tok  = test_ds.map(tokenize_batch,  batched=True, remove_columns=test_ds.column_names)

# Trainer/Seq2SeqTrainer 通常用这个 collator
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer)

print("Tokenized:", train_tok, valid_tok, test_tok)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/8000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

Map:   0%|          | 0/1283 [00:00<?, ? examples/s]

Tokenized: Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 8000
}) Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 1000
}) Dataset({
    features: ['labels', 'input_ids', 'attention_mask'],
    num_rows: 1283
})


In [11]:
# Cell 6
from transformers import AutoModelForSeq2SeqLM
from peft import (
    get_peft_model,
    LoraConfig,
    TaskType,
    PromptTuningConfig,
    PromptTuningInit
)

def build_model(cfg: dict):
    model_name = cfg.get("model_name", "facebook/bart-base")
    # 加载基础模型
    base_model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

    method = cfg.get("finetune_method", "full")
    model = None

    if method == "full":
        # 全参数微调，直接用基础模型
        model = base_model

    elif method == "lora":
        peft_config = LoraConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            inference_mode=False,
            r=16,
            lora_alpha=32,
            lora_dropout=0.1
        )
        model = get_peft_model(base_model, peft_config)

    elif method == "prompt_tuning":
        peft_config = PromptTuningConfig(
            task_type=TaskType.SEQ_2_SEQ_LM,
            num_virtual_tokens=cfg.get("vt", 20),
            prompt_tuning_init=PromptTuningInit.TEXT,
            prompt_tuning_init_text="Classify if this news is true or false:",
            tokenizer_name_or_path=model_name,
        )
        model = get_peft_model(base_model, peft_config)

    else:
        raise ValueError(f"Unknown method: {method}")

    # --- 修正部分开始：安全打印参数量 ---
    if hasattr(model, "print_trainable_parameters"):
        # 如果是 PEFT 模型 (LoRA/Prompt Tuning)
        model.print_trainable_parameters()
    else:
        # 如果是普通模型 (Full Fine-tuning)，手动计算并打印
        trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
        all_params = sum(p.numel() for p in model.parameters())
        print(f"trainable params: {trainable_params} || all params: {all_params} || trainable%: {100 * trainable_params / all_params:.2f}")
    # --- 修正部分结束 ---

    return model

In [16]:
# --- 补充缺失的 compute_metrics 函数 ---6.1
import numpy as np
from sklearn.metrics import f1_score, accuracy_score
from transformers import AutoTokenizer

# 我们需要全局 tokenizer 来解码预测出的 token ID
# 注意：这里假设你使用的是 BART base，如果换模型要改这里
global_tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")

def compute_metrics(eval_preds):
    preds, labels = eval_preds

    if isinstance(preds, tuple):
        preds = preds[0]

    # 1. 解码预测值 (Predictions)
    # 将 -100 (ignore_index) 替换为 pad_token_id 才能解码，否则会报错
    decoded_preds = global_tokenizer.batch_decode(preds, skip_special_tokens=True)

    # 2. 解码真实标签 (Labels)
    labels = np.where(labels != -100, labels, global_tokenizer.pad_token_id)
    decoded_labels = global_tokenizer.batch_decode(labels, skip_special_tokens=True)

    # 3. 文本转数值 (Post-processing)
    # 去除空格，将 "1" 转为 1， "0" 转为 0
    y_pred = []
    for text in decoded_preds:
        clean_text = text.strip()
        if clean_text == "1":
            y_pred.append(1)
        elif clean_text == "0":
            y_pred.append(0)
        else:
            # 如果模型生成了预期之外的文本（比如 "True" 或 "I think..."），算错
            # 为了计算方便，这里默认给 0 (False)，你也可以记录 invalid count
            y_pred.append(0)

    y_true = []
    for text in decoded_labels:
        clean_text = text.strip()
        try:
            y_true.append(int(clean_text))
        except:
            y_true.append(0) # 防止脏数据报错

    # 4. 计算指标
    acc = accuracy_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred, average="macro") # 重点关注 Macro-F1

    return {
        "accuracy": acc,
        "macro_f1": f1
    }

In [17]:
# Cell 7 (Fixed for Transformers v4.42+)
import torch
import time
import os
import numpy as np
from transformers import (
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

def train_model(cfg: dict, model, tokenizer, train_tok, valid_tok):
    """
    训练模型并返回评估指标和资源消耗数据
    """

    # 1. 显存清零
    if torch.cuda.is_available():
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()

    # 2. 定义 Data Collator
    data_collator = DataCollatorForSeq2Seq(
        tokenizer,
        model=model,
        label_pad_token_id=-100,
        pad_to_multiple_of=8
    )

    # 3. 准备输出目录
    run_id = cfg.get("run_id", f"run_{int(time.time())}")
    output_dir = f"checkpoints_{run_id}"

    # 4. 训练参数设置
    # 【修复点】evaluation_strategy -> eval_strategy
    args = Seq2SeqTrainingArguments(
        output_dir=output_dir,
        eval_strategy="epoch",          # 新版写法
        save_strategy="epoch",
        learning_rate=cfg["lr"],
        per_device_train_batch_size=cfg["bs"],
        per_device_eval_batch_size=cfg["bs"],
        num_train_epochs=cfg["epochs"],
        weight_decay=0.01,
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="macro_f1",
        greater_is_better=True,
        predict_with_generate=True,
        fp16=True,
        logging_dir='./logs',
        logging_steps=50,
        report_to="none"
    )

    # 5. 初始化 Trainer
    trainer = Seq2SeqTrainer(
        model=model,
        args=args,
        train_dataset=train_tok,
        eval_dataset=valid_tok,
        tokenizer=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    # 6. 开始训练
    print(f"Starting training for {run_id}...")
    start_time = time.time()

    trainer.train()

    total_time = time.time() - start_time

    # 7. 获取显存峰值
    peak_memory_mb = 0
    if torch.cuda.is_available():
        peak_memory_mb = torch.cuda.max_memory_allocated() / 1024 / 1024

    print(f"Training finished. Time: {total_time:.2f}s, Peak Memory: {peak_memory_mb:.2f} MB")

    # 8. 最后评估
    metrics = trainer.evaluate()

    metrics["train_time_sec"] = total_time
    metrics["peak_memory_mb"] = peak_memory_mb
    metrics["run_id"] = run_id

    return metrics

In [19]:
# --- 修正后的 Cell 7.1: 自动化实验循环 (含 Tokenization) ---
import pandas as pd
from transformers import AutoTokenizer

# 定义实验配置列表
experiments = [
    # 1. Full Fine-tuning (Baseline)
    {"method": "full", "lr": 5e-5, "bs": 16, "epochs": 5, "data_ratio": 1.0},

    # 2. LoRA (Main Challenger)
    {"method": "lora", "lr": 3e-4, "bs": 16, "epochs": 5, "data_ratio": 1.0},

    # 3. Prompt Tuning (Comparison)
    {"method": "prompt_tuning", "vt": 20, "lr": 0.01, "bs": 16, "epochs": 10, "data_ratio": 1.0},

    # 4. LoRA Low-Resource (Efficiency Highlight)
    {"method": "lora", "lr": 3e-4, "bs": 16, "epochs": 10, "data_ratio": 0.1},

    # 5. Full Low-Resource (To show LoRA is better at low data)
    {"method": "full", "lr": 5e-5, "bs": 16, "epochs": 10, "data_ratio": 0.1},
]

# 运行循环
results = []
for i, exp in enumerate(experiments):
    print(f"\n=== Running Experiment {i+1}/{len(experiments)}: {exp['method']} (Data: {exp['data_ratio']}) ===")

    # 0. 准备配置和 Tokenizer
    cfg = {
        "model_name": "facebook/bart-base",
        "finetune_method": exp["method"],
        "vt": exp.get("vt", 0),
        "run_id": f"exp_{i}_{exp['method']}",
        "lr": exp["lr"],
        "bs": exp["bs"],
        "epochs": exp["epochs"],
        "max_input_len": 1024, # BART 最大长度
        "max_target_len": 128
    }

    tokenizer = AutoTokenizer.from_pretrained(cfg["model_name"])

    # 1. 准备数据 (Raw Text)
    # 注意：make_hf_dataset 需要在之前的 Cell 里定义好 (Cell 5)
    current_train_df = sample_dataset(df_train, ratio=exp['data_ratio'])
    raw_train_ds = make_hf_dataset(current_train_df, "instruction")
    raw_valid_ds = make_hf_dataset(df_valid, "instruction")

    # 2. 【关键修复】数据 Tokenization
    # 必须把文本转成 input_ids，Trainer 才能跑
    def preprocess_function(examples):
        model_inputs = tokenizer(
            examples["input_text"],
            max_length=cfg["max_input_len"],
            truncation=True
        )
        labels = tokenizer(
            text_target=examples["target_text"],
            max_length=cfg["max_target_len"],
            truncation=True
        )
        model_inputs["labels"] = labels["input_ids"]
        return model_inputs

    # 批量处理
    train_ds = raw_train_ds.map(
        preprocess_function,
        batched=True,
        remove_columns=raw_train_ds.column_names, # 移除原始文本列，只保留 input_ids 等
        desc="Tokenizing train dataset"
    )
    valid_ds = raw_valid_ds.map(
        preprocess_function,
        batched=True,
        remove_columns=raw_valid_ds.column_names,
        desc="Tokenizing valid dataset"
    )

    # 3. 构建模型
    model = build_model(cfg)

    # 4. 训练
    metrics = train_model(cfg, model, tokenizer, train_ds, valid_ds)

    # 5. 记录结果
    res = {
        "method": exp["method"],
        "data_ratio": exp["data_ratio"],
        "macro_f1": metrics.get("eval_macro_f1", 0),
        "accuracy": metrics.get("eval_accuracy", 0),
        "train_time": metrics.get("train_time_sec", 0),
        "peak_memory_mb": metrics.get("peak_memory_mb", 0),
        "trainable_params": model.num_parameters(only_trainable=True)
    }
    results.append(res)

    # 实时保存
    pd.DataFrame(results).to_csv("experiment_results_final.csv", index=False)
    print(f"Result: {res}")

print("\nAll experiments finished!")
print(pd.DataFrame(results))


=== Running Experiment 1/5: full (Data: 1.0) ===


Tokenizing train dataset:   0%|          | 0/8000 [00:00<?, ? examples/s]

Tokenizing valid dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

trainable params: 139420416 || all params: 139420416 || trainable%: 100.00


/tmp/ipython-input-3131536391.py:57: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


...

Starting training for exp_4_full...


Epoch,Training Loss,Validation Loss,Accuracy,Macro F1
1,1.340900,0.232771,0.485000,0.326599
2,0.284000,0.235931,0.515000,0.339934
3,0.305100,0.223106,0.589000,0.565406
4,0.249000,0.224702,0.611000,0.604623
5,0.207400,0.240306,0.593000,0.592657
6,0.152800,0.310394,0.607000,0.606792
7,0.109500,0.379419,0.600000,0.574678
8,0.050100,0.617167,0.588000,0.572905
9,0.034600,0.782020,0.583000,0.555906
10,0.019200,0.767293,0.585000,0.567528


/usr/local/lib/python3.12/dist-packages/transformers/modeling_utils.py:3918: UserWarning: Moving the following attributes in the config to the generation config: {'early_stopping': True, 'num_beams': 4, 'no_repeat_ngram_size': 3, 'forced_bos_token_id': 0}. You are seeing this warning because you've set generation parameters in the model config, as opposed to in the generation config.
  warnings.warn(
There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Training finished. Time: 444.61s, Peak Memory: 3154.11 MB


Result: {'method': 'full', 'data_ratio': 0.1, 'macro_f1': 0.6067919929642781, 'accuracy': 0.607, 'train_time': 444.60669803619385, 'peak_memory_mb': 3154.11279296875, 'trainable_params': 139420416}

All experiments finished!
          method  data_ratio  macro_f1  accuracy  train_time  peak_memory_mb  \
0           full         1.0  0.631927     0.641  435.181805     3209.347168   
1           lora         1.0  0.613750     0.621  244.020609     2686.865723   
2  prompt_tuning         1.0  0.393560     0.497  408.259487     1842.555664   
3           lora         0.1  0.552436     0.560  143.429499     1836.372559   
4           full         0.1  0.606792     0.607  444.606698     3154.112793   

   trainable_params  
0         139420416  
1            884736  
2                 0  
3            884736  
4         139420416  


In [22]:
# --- 修正版 Cell 7.2: 自动寻找 Checkpoint 并评估 ---
import pandas as pd
import torch
import os
import glob
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Seq2SeqTrainer, Seq2SeqTrainingArguments, DataCollatorForSeq2Seq
from peft import PeftModel, PeftConfig

# 1. 定义实验目录
target_runs = [
    {"name": "Full (100% Data)", "folder": "checkpoints_exp_0_full", "method": "full"},
    {"name": "LoRA (100% Data)", "folder": "checkpoints_exp_1_lora", "method": "lora"},
    {"name": "Prompt (100% Data)", "folder": "checkpoints_exp_2_prompt_tuning", "method": "prompt"},
    {"name": "LoRA (10% Data)",  "folder": "checkpoints_exp_3_lora", "method": "lora"},
]

# --- 关键新增：自动查找 checkpoint 子目录的函数 ---
def find_best_checkpoint(folder):
    # 查找所有以 checkpoint- 开头的子文件夹
    subdirs = glob.glob(os.path.join(folder, "checkpoint-*"))
    if not subdirs:
        # 如果没有子文件夹，可能就在当前文件夹（虽然这次不是这种情况）
        return folder
    # 按时间或数字排序，选最新的一个（因为我们设置了只存最好的一个）
    # 通常 checkpoint-X，X 越大越新
    latest_ckpt = max(subdirs, key=lambda x: int(x.split("-")[-1]))
    return latest_ckpt
# ------------------------------------------------

print("=== Starting Test Set Evaluation (Fixed) ===")

# 2. 准备测试集
tokenizer = AutoTokenizer.from_pretrained("facebook/bart-base")
cfg_eval = {"max_input_len": 1024, "max_target_len": 128}

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=cfg_eval["max_input_len"],
        truncation=True
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=cfg_eval["max_target_len"],
        truncation=True
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

raw_test_ds = make_hf_dataset(df_test, "instruction")
test_ds = raw_test_ds.map(
    preprocess_function,
    batched=True,
    remove_columns=raw_test_ds.column_names,
    desc="Tokenizing test dataset"
)

# 3. 循环评估
final_results = []

for run in target_runs:
    # 自动修正路径
    actual_path = find_best_checkpoint(run["folder"])
    print(f"\nEvaluating: {run['name']}")
    print(f"   -> Looking in: {actual_path}")

    try:
        if run["method"] == "full":
            # Full Fine-tuning
            model = AutoModelForSeq2SeqLM.from_pretrained(actual_path)
        else:
            # PEFT
            base_model = AutoModelForSeq2SeqLM.from_pretrained("facebook/bart-base")
            model = PeftModel.from_pretrained(base_model, actual_path)

        if torch.cuda.is_available():
            model.to("cuda")

        # 临时 Trainer
        args = Seq2SeqTrainingArguments(
            output_dir="temp_eval_output",
            per_device_eval_batch_size=32,
            predict_with_generate=True,
            fp16=True,
            report_to="none"
        )

        trainer = Seq2SeqTrainer(
            model=model,
            args=args,
            tokenizer=tokenizer,
            data_collator=DataCollatorForSeq2Seq(tokenizer),
            compute_metrics=compute_metrics
        )

        metrics = trainer.evaluate(eval_dataset=test_ds)

        final_results.append({
            "Model Variant": run["name"],
            "Test Macro-F1": metrics["eval_macro_f1"],
            "Test Accuracy": metrics["eval_accuracy"]
        })
        print(f"   -> Success! F1={metrics['eval_macro_f1']:.4f}")

    except Exception as e:
        print(f"   !!! Failed: {e}")

# 4. 结果展示
print("\n" + "="*30)
print("=== FINAL PAPER TABLE ===")
print("="*30)
df_final = pd.DataFrame(final_results)
print(df_final)

df_final.to_csv("final_paper_results_test_set.csv", index=False)

=== Starting Test Set Evaluation (Fixed) ===


Tokenizing test dataset:   0%|          | 0/1283 [00:00<?, ? examples/s]


Evaluating: Full (100% Data)
   -> Looking in: checkpoints_exp_0_full/checkpoint-2500


/tmp/ipython-input-1213787919.py:88: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


   -> Success! F1=0.5807

Evaluating: LoRA (100% Data)
   -> Looking in: checkpoints_exp_1_lora/checkpoint-2500


...

   -> Success! F1=0.6022

Evaluating: Prompt (100% Data)
   -> Looking in: checkpoints_exp_2_prompt_tuning/checkpoint-2000


/tmp/ipython-input-1213787919.py:88: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


   -> Success! F1=0.4049

Evaluating: LoRA (10% Data)
   -> Looking in: checkpoints_exp_3_lora/checkpoint-300


/tmp/ipython-input-1213787919.py:88: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Seq2SeqTrainer.__init__`. Use `processing_class` instead.
  trainer = Seq2SeqTrainer(


   -> Success! F1=0.5400

=== FINAL PAPER TABLE ===
        Model Variant  Test Macro-F1  Test Accuracy
0    Full (100% Data)       0.580654       0.605612
1    LoRA (100% Data)       0.602152       0.619641
2  Prompt (100% Data)       0.404857       0.472330
3     LoRA (10% Data)       0.540033       0.555729


In [20]:
# Cell 8
@torch.inference_mode()
def generate_with_token_counts(model, tokenizer, inputs, max_new_tokens=64, batch_size=16):
    """
    返回：
    - texts: List[str]
    - gen_token_counts: List[int]  # 按 decoded text 的 subword token 数统计（不含 special tokens）
    """
    device = next(model.parameters()).device
    texts = []
    gen_token_counts = []

    for i in range(0, len(inputs), batch_size):
        batch = inputs[i:i+batch_size]
        toks = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=CONFIG["max_input_len"],   # ✅ 修 warning & 统一输入长度
        ).to(device)

        gen_ids = model.generate(
          **toks,
          do_sample=False,
          num_beams=1,
          max_new_tokens=min(max_new_tokens, 4),
          min_new_tokens=1,
          eos_token_id=tokenizer.eos_token_id,
          pad_token_id=tokenizer.pad_token_id,
      )


        batch_texts = tokenizer.batch_decode(gen_ids, skip_special_tokens=True)

        # ✅ 用 tokenizer 编码 decoded text 再计数：跨模型一致
        for t in batch_texts:
            gen_token_counts.append(len(tokenizer.encode(t, add_special_tokens=False)))

        texts.extend(batch_texts)

    return texts, gen_token_counts


def evaluate(cfg: dict, ckpt_dir: str, test_df: pd.DataFrame):
    tok = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)

    method = cfg.get("finetune_method", "full")

    if method in ("prompt", "lora"):
        base = AutoModelForSeq2SeqLM.from_pretrained(cfg["model_name"])
        model = PeftModel.from_pretrained(base, ckpt_dir)
    else:
        model = AutoModelForSeq2SeqLM.from_pretrained(ckpt_dir)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()

    inputs = [build_prompt(s, cfg["run_type"]) for s in test_df["statement"].tolist()]
    true_labels = test_df["labels"].astype(int).tolist()

    batch_size = (16 if device == "cuda" else 4)
    texts, gen_token_counts = generate_with_token_counts(
        model, tok, inputs,
        max_new_tokens=cfg["max_new_tokens"],
        batch_size=batch_size
    )
    pred_labels = [extract_label(t) for t in texts]

    mf1 = f1_score(true_labels, pred_labels, average="macro")
    acc = accuracy_score(true_labels, pred_labels)
    avg_gen_tokens = float(np.mean(gen_token_counts)) if len(gen_token_counts) else 0.0

    n_lat = min(300, len(inputs))
    sample_inputs = inputs[:n_lat]

    _ = generate_with_token_counts(model, tok, sample_inputs[:min(8, n_lat)], max_new_tokens=cfg["max_new_tokens"], batch_size=1)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    t0 = time.time()
    _ = generate_with_token_counts(model, tok, sample_inputs, max_new_tokens=cfg["max_new_tokens"], batch_size=1)
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    latency_ms = (time.time() - t0) / n_lat * 1000.0

    return {
        "macro_f1": float(mf1),
        "accuracy": float(acc),
        "avg_generated_tokens": float(avg_gen_tokens),
        "latency_ms_per_sample": float(latency_ms),
        "n_test": int(len(inputs)),
        "n_latency_samples": int(n_lat),
    }



metrics = evaluate(CONFIG, ckpt_dir, test_b)
metrics


NameError: name 'ckpt_dir' is not defined

In [ ]:
# Cell 9
def get_dir_size_mb(path: str) -> float:
    total = 0
    for root, _, files in os.walk(path):
        for fn in files:
            fp = os.path.join(root, fn)
            if os.path.isfile(fp):
                total += os.path.getsize(fp)
    return total / (1024**2)

def safe_div(a, b):
    return float(a) / float(b) if b not in [0, 0.0, None] else None

def count_params_safe():
    """
    Return (total_params, trainable_params, trainable_ratio) or (None, None, None)
    if `model` doesn't exist in the global scope.
    """
    if "model" not in globals() or globals().get("model") is None:
        return None, None, None
    m = globals()["model"]
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    ratio = (trainable / total) if total else None
    return int(total), int(trainable), ratio

stamp = time.strftime("%Y%m%d_%H%M%S")
out_path = os.path.join(CONFIG["artifacts_dir"], f"manifest_{CONFIG['run_type']}_{stamp}.json")

# --- 训练时间规范化 ---
n_train = int(len(train_b))
sec_total = float(train_time)  # 你前面训练阶段算出来的 train_time
sec_per_epoch = safe_div(sec_total, CONFIG["epochs"])
sec_per_1k = safe_div(sec_total, (n_train / 1000.0))

# --- 性价比（Macro-F1 / latency_ms）---
f1_per_ms = safe_div(metrics.get("macro_f1"), metrics.get("latency_ms_per_sample"))
f1_per_s  = None if f1_per_ms is None else f1_per_ms * 1000.0

# --- 参数统计（对 LoRA / prompt tuning 特别关键）---
total_p, trainable_p, trainable_ratio = count_params_safe()

manifest = {
    "config": CONFIG,
    "ckpt_dir": ckpt_dir,

    # ✅ 兼容字段：让旧的 audit / summary 直接读到（你现在 Cell 18 就是读这个）
    "train_time_sec": sec_total,

    # （可选但很有用）也放一份顶层规范化字段，方便别的脚本读取
    "train_time_sec_per_epoch": sec_per_epoch,
    "train_time_sec_per_1k_train_samples": sec_per_1k,

    # 你关心的核心指标（1）
    "metrics": metrics,

    # 你关心的训练成本（总训练时间 + 规范化）（5）
    "training_cost": {
        "train_time_sec_total": sec_total,
        "train_time_sec_per_epoch": sec_per_epoch,
        "train_time_sec_per_1k_train_samples": sec_per_1k,
        "n_train_samples": n_train,
    },

    # 性价比（2）
    "efficiency": {
        "macro_f1_per_ms": f1_per_ms,
        "macro_f1_per_s": f1_per_s,
    },

    # ✅ 新增：论文里很好写的“可训练参数占比”
    "params": {
        "total_params": total_p,
        "trainable_params": trainable_p,
        "trainable_ratio": trainable_ratio,
    },

    # 工程记录（可删）
    "artifact_info": {
        "checkpoint_size_mb": float(get_dir_size_mb(ckpt_dir)),
    },
}

with open(out_path, "w", encoding="utf-8") as f:
    json.dump(manifest, f, ensure_ascii=False, indent=2)

print("[SAVE CHECK] artifacts_dir =", CONFIG["artifacts_dir"])
print("Saved manifest:", out_path)

print("\n=== RUN SUMMARY ===")
print("model_name:", CONFIG["model_name"])
print("run_type  :", CONFIG["run_type"])
print("finetune_method:", CONFIG.get("finetune_method", "full/unknown"))
print("num_virtual_tokens:", CONFIG.get("num_virtual_tokens", None))

print("Macro-F1  :", manifest["metrics"]["macro_f1"])
print("Accuracy  :", manifest["metrics"]["accuracy"])
print("Latency ms/sample:", manifest["metrics"]["latency_ms_per_sample"])
print("Avg gen tokens   :", manifest["metrics"].get("avg_generated_tokens", None))
print("Train time (sec) :", manifest["training_cost"]["train_time_sec_total"])
print("Train sec/epoch  :", manifest["training_cost"]["train_time_sec_per_epoch"])
print("Train sec/1k     :", manifest["training_cost"]["train_time_sec_per_1k_train_samples"])
print("F1 per second    :", manifest["efficiency"]["macro_f1_per_s"])

print("Total params     :", manifest["params"]["total_params"])
print("Trainable params :", manifest["params"]["trainable_params"])
print("Trainable ratio  :", manifest["params"]["trainable_ratio"])


In [ ]:
# Cell 10
# 汇总 artifacts_dir 里所有 manifest_*.json -> results dataframe，并保存 csv
import glob

def load_manifests(artifacts_dir):
    paths = sorted(glob.glob(os.path.join(artifacts_dir, "manifest_*.json")))
    runs = []
    for p in paths:
        with open(p, "r", encoding="utf-8") as f:
            m = json.load(f)

        cfg = m["config"]
        met = m["metrics"]
        tc  = m.get("training_cost", {})
        eff = m.get("efficiency", {})

        runs.append({
            "manifest": os.path.basename(p),
            "model_name": cfg.get("model_name"),
            "run_type": cfg.get("run_type"),
            "seed": cfg.get("seed"),
            "epochs": cfg.get("epochs"),
            "train_limit": cfg.get("train_limit"),
            "test_limit": cfg.get("test_limit"),

            # (1) 直接指标
            "macro_f1": met.get("macro_f1"),
            "accuracy": met.get("accuracy"),
            "latency_ms_per_sample": met.get("latency_ms_per_sample"),
            "avg_generated_tokens": met.get("avg_generated_tokens"),

            # (5) 训练成本
            "train_time_sec_total": tc.get("train_time_sec_total"),
            "train_time_sec_per_epoch": tc.get("train_time_sec_per_epoch"),
            "train_time_sec_per_1k_train_samples": tc.get("train_time_sec_per_1k_train_samples"),

            # (2) 性价比
            "macro_f1_per_s": eff.get("macro_f1_per_s"),
        })
    return pd.DataFrame(runs)

df_results = load_manifests(CONFIG["artifacts_dir"])

# 按“性价比”排序（你更关心哪列可以换）
df_results_sorted = df_results.sort_values(
    by=["macro_f1_per_s", "macro_f1"],
    ascending=False
).reset_index(drop=True)

csv_path = os.path.join(CONFIG["artifacts_dir"], "results_summary.csv")
df_results_sorted.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df_results_sorted


In [ ]:
# Cell 11
import re

def parse_stamp_from_manifest_name(name: str):
    # manifest_instruction_YYYYMMDD_HHMMSS.json
    m = re.search(r"_(\d{8})_(\d{6})\.json$", name)
    if not m:
        return ""
    return m.group(1) + m.group(2)  # sortable string

df = df_results.copy()

# 1) 标记字段是否完整（你关心的 1/2/5 指标都齐）
needed_cols = ["avg_generated_tokens", "train_time_sec_total", "macro_f1_per_s"]
df["is_complete"] = df[needed_cols].notna().all(axis=1)

# 2) 提取时间戳（用于“最新优先”）
df["stamp"] = df["manifest"].apply(parse_stamp_from_manifest_name)

# 3) 排序规则：同组 (model,run_type) 下，优先 complete，其次最新
df = df.sort_values(
    by=["model_name", "run_type", "is_complete", "stamp"],
    ascending=[True, True, False, False]
)

# 4) 每组只保留一条（最新且完整优先）
df_best = df.drop_duplicates(subset=["model_name", "run_type"], keep="first").reset_index(drop=True)

# 5) 输出论文友好表（你要的 1/2/5）
paper_cols = [
    "model_name", "run_type",
    "macro_f1", "accuracy",
    "latency_ms_per_sample", "avg_generated_tokens",
    "train_time_sec_total", "train_time_sec_per_epoch", "train_time_sec_per_1k_train_samples",
    "macro_f1_per_s",
    "manifest"
]
df_paper = df_best[paper_cols].sort_values(by=["macro_f1_per_s", "macro_f1"], ascending=False).reset_index(drop=True)

csv_path = os.path.join(CONFIG["artifacts_dir"], "results_paper_best.csv")
df_paper.to_csv(csv_path, index=False)

print("Saved:", csv_path)
df_paper


In [ ]:
# Cell 12
import random
import pandas as pd
import numpy as np
from sklearn.metrics import confusion_matrix, classification_report

def majority_baseline(df):
    y = df["labels"].astype(int).tolist()
    p0 = [0] * len(y)  # always False
    p1 = [1] * len(y)  # always True
    f1_0 = f1_score(y, p0, average="macro")
    f1_1 = f1_score(y, p1, average="macro")
    acc_0 = accuracy_score(y, p0)
    acc_1 = accuracy_score(y, p1)
    return {
        "always_false": {"macro_f1": float(f1_0), "accuracy": float(acc_0)},
        "always_true":  {"macro_f1": float(f1_1), "accuracy": float(acc_1)},
    }

print("=== Majority baselines on current test_b ===")
print(majority_baseline(test_b))

# ---- 误差分析：抽样看模型原始输出 & 解析结果 ----
def sample_generations_for_debug(cfg, ckpt_dir, df, k=20, seed=42):
    rnd = random.Random(seed)
    idxs = list(range(len(df)))
    rnd.shuffle(idxs)
    idxs = idxs[:min(k, len(df))]

    # 复用你 Cell 8 的 evaluate() 那套加载方式
    tok = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)
    if cfg["run_type"] == "prompt":
        base = AutoModelForSeq2SeqLM.from_pretrained(cfg["model_name"])
        m = PeftModel.from_pretrained(base, ckpt_dir)
    else:
        m = AutoModelForSeq2SeqLM.from_pretrained(ckpt_dir)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    m.to(device)
    m.eval()

    rows = []
    for i in idxs:
        stmt = df.loc[i, "statement"]
        y = int(df.loc[i, "labels"])
        inp = build_prompt(stmt, cfg["run_type"])

        texts, token_counts = generate_with_token_counts(
            m, tok, [inp],
            max_new_tokens=cfg["max_new_tokens"],
            batch_size=1
        )
        gen_text = texts[0]
        pred = extract_label(gen_text)

        rows.append({
            "true_label": y,
            "true_str": label_to_str(y),
            "pred_label": pred,
            "pred_str": label_to_str(pred),
            "gen_tokens": token_counts[0],
            "statement": stmt,
            "generated": gen_text
        })

    return pd.DataFrame(rows)

debug_df = sample_generations_for_debug(CONFIG, ckpt_dir, test_b, k=20, seed=CONFIG["seed"])
debug_df


In [ ]:
# Cell 13
# 重新跑一次得到 pred_labels（复用你 Cell 8 evaluate 的逻辑，但输出 preds）
def get_predictions(cfg, ckpt_dir, test_df):
    tok = AutoTokenizer.from_pretrained(ckpt_dir, use_fast=True)
    if cfg["run_type"] == "prompt":
        base = AutoModelForSeq2SeqLM.from_pretrained(cfg["model_name"])
        m = PeftModel.from_pretrained(base, ckpt_dir)
    else:
        m = AutoModelForSeq2SeqLM.from_pretrained(ckpt_dir)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    m.to(device)
    m.eval()

    inputs = [build_prompt(s, cfg["run_type"]) for s in test_df["statement"].tolist()]
    true_labels = test_df["labels"].astype(int).tolist()

    texts, _ = generate_with_token_counts(
        m, tok, inputs,
        max_new_tokens=cfg["max_new_tokens"],
        batch_size=(16 if device=="cuda" else 4)
    )
    pred_labels = [extract_label(t) for t in texts]
    return true_labels, pred_labels

y_true, y_pred = get_predictions(CONFIG, ckpt_dir, test_b)

print("Confusion matrix [ [TN, FP], [FN, TP] ]:")
print(confusion_matrix(y_true, y_pred))

print("\nClassification report:")
print(classification_report(y_true, y_pred, target_names=["False(0)", "True(1)"], digits=4))


In [ ]:
# Cell 14 — AUDIT (no training, no re-eval)
import os, json, glob, math
import pandas as pd

ART_DIR = CONFIG.get("artifacts_dir", "artifacts_liar_v1")
print("ARTIFACTS_DIR =", ART_DIR)

# 1) 列出 manifests
manifests = sorted(glob.glob(os.path.join(ART_DIR, "manifest_*.json")))
print(f"Found {len(manifests)} manifests")
for m in manifests[-8:]:
    print("  ", os.path.basename(m))

# 2) 读 manifests 形成 dataframe（检查缺字段/版本漂移）
rows = []
for p in manifests:
    try:
        d = json.load(open(p, "r"))
    except Exception as e:
        print("Failed to read:", p, "err:", e)
        continue
    cfg = d.get("config", {})
    met = d.get("metrics", {})
    row = {
        "manifest": os.path.basename(p),
        "model_name": cfg.get("model_name"),
        "run_type": cfg.get("run_type"),
        "seed": cfg.get("seed"),
        "epochs": cfg.get("epochs"),
        "lr": cfg.get("lr"),
        "train_limit": cfg.get("train_limit"),
        "test_limit": cfg.get("test_limit"),
        "macro_f1": met.get("macro_f1"),
        "accuracy": met.get("accuracy"),
        "latency_ms_per_sample": met.get("latency_ms_per_sample"),
        "avg_generated_tokens": met.get("avg_generated_tokens"),
        "train_time_sec_total": d.get("train_time_sec"),
        "ckpt_dir": d.get("ckpt_dir"),
    }
    # derived efficiency: F1 per second (与你 summary 一致)
    lat = row["latency_ms_per_sample"]
    f1 = row["macro_f1"]
    row["macro_f1_per_s"] = (f1 / lat * 1000.0) if (isinstance(lat,(int,float)) and lat and isinstance(f1,(int,float))) else None
    rows.append(row)

df = pd.DataFrame(rows)
print("\n=== MANIFESTS (tail) ===")
display(df.tail(10))

# 3) 检查“评测口径漂移”的高风险信号
issues = []

# 3.1 缺字段
for col in ["avg_generated_tokens","latency_ms_per_sample","macro_f1"]:
    n_missing = df[col].isna().sum()
    if n_missing:
        issues.append(f"[Missing] {col}: {n_missing} rows missing")

# 3.2 极端 token/latency（可能是未cap/输出失控）
if "avg_generated_tokens" in df.columns:
    bad_tok = df[df["avg_generated_tokens"].fillna(0) > 10][["manifest","model_name","run_type","avg_generated_tokens","latency_ms_per_sample"]]
    if len(bad_tok):
        issues.append("[Outlier] avg_generated_tokens > 10 exists (likely uncontrolled decoding)")
        display(bad_tok)

# 3.3 近似 baseline（提示塌缩）
# 你当前 test_b 的 baseline（你之前贴过）
BASE_FALSE_F1 = 0.30233822729744425
BASE_TRUE_F1  = 0.3616915422885572
near_base = df[df["macro_f1"].apply(lambda x: isinstance(x,(int,float)) and (abs(x-BASE_FALSE_F1)<1e-3 or abs(x-BASE_TRUE_F1)<1e-3))][
    ["manifest","model_name","run_type","macro_f1","accuracy","avg_generated_tokens","latency_ms_per_sample"]
]
if len(near_base):
    issues.append("[Degenerate] macro_f1 equals (or extremely close to) majority baseline on some runs")
    display(near_base)

print("\n=== AUDIT ISSUES ===")
if not issues:
    print("No obvious issues flagged from manifests.")
else:
    for s in issues:
        print("-", s)

# 4) 给你一个“论文主表”候选（每个 model+run_type 取 macro_f1 最大的那条）
print("\n=== CANDIDATE BEST PER (model_name, run_type) ===")
best = (df.sort_values(["model_name","run_type","macro_f1"], ascending=[True,True,False])
          .groupby(["model_name","run_type"], as_index=False)
          .head(1))
display(best[[
    "model_name","run_type","macro_f1","accuracy","latency_ms_per_sample","avg_generated_tokens",
    "train_time_sec_total","macro_f1_per_s","manifest","ckpt_dir"
]])


In [ ]:
# Cell 15 — find all manifests in this runtime
!pwd
!ls -lah
!find /content -name "manifest_*.json" -type f 2>/dev/null | sed 's|^|FOUND: |'


In [ ]:
# Cell 17 — Step2: Build paper tables (main + ablation) and save CSV
import os, json, glob
import pandas as pd

ART_DIR = CONFIG["artifacts_dir"]
print("Scanning artifacts_dir:", ART_DIR)

manifest_paths = sorted(glob.glob(os.path.join(ART_DIR, "manifest_*.json")))
print("Found manifests:", len(manifest_paths))
assert len(manifest_paths) > 0, "❌ 没找到任何 manifest_*.json，请先完成 Step1 把文件都放进 Drive artifacts_dir"

rows = []
for p in manifest_paths:
    with open(p, "r") as f:
        d = json.load(f)
    cfg = d.get("config", {})
    met = d.get("metrics", {})
    rows.append({
        "manifest": os.path.basename(p),
        "ckpt_dir": d.get("ckpt_dir"),
        "model_name": cfg.get("model_name"),
        "run_type": cfg.get("run_type"),
        "num_virtual_tokens": cfg.get("num_virtual_tokens"),
        "seed": cfg.get("seed"),
        "epochs": cfg.get("epochs"),
        "lr": cfg.get("lr"),
        "train_limit": cfg.get("train_limit"),
        "test_limit": cfg.get("test_limit"),
        "macro_f1": met.get("macro_f1"),
        "accuracy": met.get("accuracy"),
        "latency_ms_per_sample": met.get("latency_ms_per_sample"),
        "avg_generated_tokens": met.get("avg_generated_tokens"),
        "train_time_sec_total": d.get("train_time_sec"),
    })

df = pd.DataFrame(rows)

# efficiency score: F1 per second（你现在打印的那个）
df["macro_f1_per_s"] = df.apply(
    lambda r: (r["macro_f1"] / r["latency_ms_per_sample"] * 1000.0)
    if pd.notnull(r["macro_f1"]) and pd.notnull(r["latency_ms_per_sample"]) and r["latency_ms_per_sample"] > 0
    else None,
    axis=1
)

# -------- Main table（主表：3 行）--------
# 取这三种设置里 macro_f1 最高的一条（通常每种你只有 1 条）
main_candidates = df[
    ((df["model_name"] == "facebook/bart-base") & (df["run_type"] == "instruction")) |
    ((df["model_name"] == "google/flan-t5-small") & (df["run_type"] == "instruction")) |
    ((df["model_name"] == "facebook/bart-base") & (df["run_type"] == "prompt") & (df["num_virtual_tokens"] == 20))
].copy()

main_table = (main_candidates
              .sort_values(["model_name","run_type","macro_f1"], ascending=[True,True,False])
              .drop_duplicates(["model_name","run_type","num_virtual_tokens"], keep="first"))

main_cols = [
    "model_name","run_type","num_virtual_tokens",
    "macro_f1","accuracy","latency_ms_per_sample","avg_generated_tokens",
    "train_time_sec_total","macro_f1_per_s","manifest"
]
main_table = main_table[main_cols].reset_index(drop=True)

# -------- Ablation table（prompt vt=10/20/30/50）--------
abl_table = df[
    (df["model_name"] == "facebook/bart-base") &
    (df["run_type"] == "prompt") &
    (df["num_virtual_tokens"].isin([10,20,30,50]))
].copy()

abl_cols = [
    "num_virtual_tokens",
    "macro_f1","accuracy","latency_ms_per_sample","avg_generated_tokens",
    "train_time_sec_total","macro_f1_per_s","manifest"
]
abl_table = (abl_table.sort_values("num_virtual_tokens")[abl_cols]
             .reset_index(drop=True))

# 保存 CSV
out_main = os.path.join(ART_DIR, "paper_main_table.csv")
out_abl  = os.path.join(ART_DIR, "paper_prompt_ablation.csv")
main_table.to_csv(out_main, index=False)
abl_table.to_csv(out_abl, index=False)

print("\n✅ Saved:", out_main)
display(main_table)

print("\n✅ Saved:", out_abl)
display(abl_table)


In [ ]:
# Cell 18
import os, glob

DRIVE_ART_DIR = "/content/drive/MyDrive/liar_runs/artifacts_liar_v1"
TARGET = "manifest_instruction_20251208_073438.json"

print("DRIVE_ART_DIR:", DRIVE_ART_DIR)
drive_hits = glob.glob(os.path.join(DRIVE_ART_DIR, TARGET))
print("Drive hits:", drive_hits)

# 1) 先查 /content 下所有位置（包含你本地 artifacts_liar_v1）
content_hits = glob.glob(os.path.join("/content", "**", TARGET), recursive=True)
print("\n/content hits:", len(content_hits))
for p in content_hits[:50]:
    print(" -", p)

# 2) 再查 Drive 更大范围（避免你把它写到了别的文件夹）
drive_wide_hits = glob.glob(os.path.join("/content/drive/MyDrive", "**", TARGET), recursive=True)
print("\nDrive-wide hits:", len(drive_wide_hits))
for p in drive_wide_hits[:50]:
    print(" -", p)

if not drive_hits and not content_hits and not drive_wide_hits:
    print("\n⚠️ 没找到这个文件：说明它那次 run 的 manifest 没成功写出，或文件名不是这个。")


In [ ]:
# Cell 18.1
import os, shutil

DRIVE_ART_DIR = "/content/drive/MyDrive/liar_runs/artifacts_liar_v1"
src = os.path.join(DRIVE_ART_DIR, "archive_duplicates", "manifest_instruction_20251208_073438.json")
dst = os.path.join(DRIVE_ART_DIR, os.path.basename(src))

assert os.path.exists(src), f"Not found: {src}"
shutil.copy2(src, dst)  # 用 copy，避免破坏 archive
print("Restored copy to:", dst)


In [ ]:
# Cell 19 — Dedupe manifests by setting, keep best efficiency (eff_f1_per_s)

import os, glob, json
import pandas as pd

ART_DIR = DRIVE_ART_DIR  # 你前面定义的 Drive 目录
manifest_paths = sorted(glob.glob(os.path.join(ART_DIR, "manifest_*.json")))

def _safe_get(d, *keys, default=None):
    cur = d
    for k in keys:
        if not isinstance(cur, dict) or k not in cur:
            return default
        cur = cur[k]
    return cur

rows = []
for p in manifest_paths:
    with open(p, "r", encoding="utf-8") as f:
        m = json.load(f)

    cfg = m.get("config", {})
    metrics = m.get("metrics", {})
    eff = m.get("efficiency", {})

    model_name = cfg.get("model_name")
    run_type = cfg.get("run_type")
    vt = int(cfg.get("num_virtual_tokens", 0) or 0)
    epochs = int(cfg.get("epochs", 0) or 0)
    lr = float(cfg.get("lr", 0) or 0)

    # ---- finetune_method fallback (兼容旧manifest) ----
    ftm = cfg.get("finetune_method", None)
    if ftm is None:
        if run_type == "prompt":
            ftm = "prompt_tuning"
        else:
            # instruction 的旧跑法默认 full
            ftm = "full"

    # ---- efficiency fallback (兼容旧manifest) ----
    macro_f1 = float(metrics.get("macro_f1", 0) or 0)
    latency = float(metrics.get("latency_ms_per_sample", 0) or 0)
    eff_f1_per_s = eff.get("macro_f1_per_s", None)
    if eff_f1_per_s is None and latency > 0:
        eff_f1_per_s = macro_f1 / latency * 1000.0

    rows.append({
        "manifest_path": p,
        "manifest": os.path.basename(p),
        "model_name": model_name,
        "run_type": run_type,
        "vt": vt,
        "epochs": epochs,
        "lr": lr,
        "finetune_method": ftm,
        "macro_f1": macro_f1,
        "accuracy": float(metrics.get("accuracy", 0) or 0),
        "latency_ms_per_sample": latency,
        "avg_generated_tokens": float(metrics.get("avg_generated_tokens", 0) or 0),
        "eff_f1_per_s": float(eff_f1_per_s or 0),
        "train_time_sec_total": float(_safe_get(m, "training_cost", "train_time_sec_total", default=_safe_get(m, "train_time_sec", default=0)) or 0),
        "trainable_params": int(m.get("trainable_params", _safe_get(m, "config", "trainable_params", default=0)) or 0),
        "total_params": int(m.get("total_params", _safe_get(m, "config", "total_params", default=0)) or 0),
        "trainable_ratio": float(m.get("trainable_ratio", _safe_get(m, "config", "trainable_ratio", default=0)) or 0),
    })

df_all = pd.DataFrame(rows)

# key：把 finetune_method 也纳入 setting
setting_cols = ["model_name", "run_type", "vt", "epochs", "lr", "finetune_method"]

# 同 setting 多次，保留 eff_f1_per_s 最大的
df_best = (
    df_all.sort_values("eff_f1_per_s", ascending=False)
          .drop_duplicates(subset=setting_cols, keep="first")
          .reset_index(drop=True)
)

print("ART_DIR =", ART_DIR)
print("Found manifests:", len(df_all))
print("After dedupe per setting:", len(df_best))

# 给 Cell20 用
DF_MANIFESTS_DEDUPED = df_best


In [ ]:
# Cell 20 — Build 3-way main table (Full vs Prompt-tuning vs LoRA) and save new CSVs (no overwrite)

import os
import pandas as pd

df = DF_MANIFESTS_DEDUPED.copy()

# 你要的三行：Instruction(full) vs Prompt-tuning(vt=20) vs Instruction(LoRA)
target_rows = []

# 1) baseline: instruction + full FT + bart-base
target_rows.append(
    df[(df["model_name"]=="facebook/bart-base") &
       (df["run_type"]=="instruction") &
       (df["finetune_method"]=="full")]
)

# 2) prompt-tuning: prompt + vt=20 + bart-base
target_rows.append(
    df[(df["model_name"]=="facebook/bart-base") &
       (df["run_type"]=="prompt") &
       (df["finetune_method"]=="prompt_tuning") &
       (df["vt"]==20)]
)

# 3) LoRA: instruction + lora + bart-base
target_rows.append(
    df[(df["model_name"]=="facebook/bart-base") &
       (df["run_type"]=="instruction") &
       (df["finetune_method"]=="lora")]
)

df_3 = pd.concat(target_rows, axis=0)

# 如果某类有多条（比如你重复跑 baseline），再按效率保留一条
setting_cols = ["model_name", "run_type", "vt", "epochs", "lr", "finetune_method"]
df_3 = (df_3.sort_values("eff_f1_per_s", ascending=False)
            .drop_duplicates(subset=setting_cols, keep="first")
            .reset_index(drop=True))

# 论文展示字段（你可以再按需加/删）
out = df_3[[
    "model_name","run_type","finetune_method","vt","epochs","lr",
    "macro_f1","accuracy","latency_ms_per_sample","eff_f1_per_s",
    "train_time_sec_total","trainable_params","total_params","trainable_ratio",
    "manifest"
]].copy()

# instruction 的 vt 显示成 "-"
out.loc[out["run_type"]=="instruction", "vt"] = "-"

print("\n=== PAPER MAIN TABLE (3-way) ===")
display(out)

# 保存：单独命名，不覆盖你原来的 csv
save_path = os.path.join(DRIVE_ART_DIR, "results_paper_main_3way.csv")
out.to_csv(save_path, index=False)
print("\nSaved:", save_path, "| bytes:", os.path.getsize(save_path))

# 也把这个 df 暴露出来，后面画图用
DF_PAPER_MAIN_3WAY = out


In [ ]:
  # Cell 21 — Make 4 paper figures from DF_PAPER_MAIN_3WAY (matplotlib only)

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

assert "DF_PAPER_MAIN_3WAY" in globals(), "❌ DF_PAPER_MAIN_3WAY 不存在：请先跑 Cell20"

dfp = DF_PAPER_MAIN_3WAY.copy()

# ---- 给每行取一个论文里好看的方法名（固定顺序）----
def method_label(row):
    if row["run_type"] == "instruction" and row["finetune_method"] == "full":
        return "Instruction (Full FT)"
    if row["run_type"] == "prompt" and row["finetune_method"] == "prompt_tuning":
        # vt 可能是字符串或数字
        vt = row["vt"]
        return f"Prompt-tuning (vt={vt})"
    if row["run_type"] == "instruction" and row["finetune_method"] == "lora":
        return "Instruction (LoRA)"
    return f'{row["run_type"]}-{row["finetune_method"]}'

dfp["method"] = dfp.apply(method_label, axis=1)

order = ["Instruction (Full FT)", "Prompt-tuning (vt=20)", "Instruction (LoRA)"]
dfp["method"] = pd.Categorical(dfp["method"], categories=order, ordered=True)
dfp = dfp.sort_values("method")

# ---- 数值列确保是 float/int ----
num_cols = ["macro_f1","accuracy","latency_ms_per_sample","eff_f1_per_s",
            "train_time_sec_total","trainable_params","total_params","trainable_ratio"]
for c in num_cols:
    if c in dfp.columns:
        dfp[c] = pd.to_numeric(dfp[c], errors="coerce")

display(dfp[["model_name","run_type","finetune_method","vt","epochs","lr",
             "macro_f1","accuracy","latency_ms_per_sample","eff_f1_per_s",
             "train_time_sec_total","trainable_ratio","trainable_params","manifest"]])

# ---- 保存设置（可选）----
SAVE_FIGS = True
FIG_DIR = os.path.join(DRIVE_ART_DIR, "paper_figs")
os.makedirs(FIG_DIR, exist_ok=True)

x = np.arange(len(dfp))
labels = dfp["method"].astype(str).tolist()

def _save(fig, name):
    if SAVE_FIGS:
        path_png = os.path.join(FIG_DIR, f"{name}.png")
        fig.savefig(path_png, dpi=300, bbox_inches="tight")
        print("Saved:", path_png)

# ========== Fig 1: Macro-F1 ==========
fig = plt.figure()
plt.bar(x, dfp["macro_f1"].values)
plt.xticks(x, labels, rotation=15, ha="right")
plt.ylabel("Macro-F1")
plt.title("Performance (Macro-F1)")
plt.tight_layout()
_save(fig, "fig1_macro_f1")
plt.show()

# ========== Fig 2: Latency ==========
fig = plt.figure()
plt.bar(x, dfp["latency_ms_per_sample"].values)
plt.xticks(x, labels, rotation=15, ha="right")
plt.ylabel("Latency (ms / sample)")
plt.title("Inference Latency")
plt.tight_layout()
_save(fig, "fig2_latency_ms")
plt.show()

# ========== Fig 3: Efficiency (F1 per second) ==========
fig = plt.figure()
plt.bar(x, dfp["eff_f1_per_s"].values)
plt.xticks(x, labels, rotation=15, ha="right")
plt.ylabel("Efficiency (Macro-F1 per second)")
plt.title("Efficiency (Macro-F1 / latency)")
plt.tight_layout()
_save(fig, "fig3_eff_f1_per_s")
plt.show()

# ========== Fig 4: Trainable ratio (or trainable params) ==========
# 你文章里更推荐 ratio（更直观说明“成本”），如果你想换成 trainable_params，我也能给你换
fig = plt.figure()
plt.bar(x, dfp["trainable_ratio"].values)
plt.xticks(x, labels, rotation=15, ha="right")
plt.ylabel("Trainable Ratio")
plt.title("Trainable Parameter Ratio")
plt.tight_layout()
_save(fig, "fig4_trainable_ratio")
plt.show()

print("\nFIG_DIR =", FIG_DIR)
